# Baseline: K-Means on Raw Graph Features

This notebook trains a K-Means clustering model directly on the **raw root-node features**
extracted from the pre-built dependency graph dataset — without passing them through a GNN.

It serves as a graph-aware baseline to compare against the GCL-trained GNN model in
`train_gnn_model.ipynb`. Both notebooks use the same graph dataset, the same train/test
split seed, and the same evaluation metrics, so results are directly comparable.

## Workflow
1. Load the cached PyG graph dataset from `data/gnn_graph_dataset.pkl`
2. Extract the **root-node raw feature vector** (`data.x[0]`, dim=27) for each package
3. Apply the same train/test split as the GNN notebook (80/20, seed=42)
4. Standardise features and fit K-Means on the **test set** using the Elbow Method
5. Evaluate: risk score per cluster → top-K% clusters → Recall / Precision / F1

## Setup

In [1]:
import sys
import os
import pickle
import random

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import joblib

# ── Paths ──────────────────────────────────────────────────────────────────────────
GRAPH_CACHE   = '../data/gnn_graph_dataset.pkl'
MODEL_OUTPUT  = '../models/lib/raw_graph_kmeans.pkl'
SCALER_OUTPUT = '../models/lib/raw_graph_scaler.pkl'

# ── Hyper-parameters (match train_gnn_model.ipynb) ─────────────────────────────────────
TRAIN_RATIO = 0.8
TOP_K_PCT   = 0.33

os.makedirs('../models/lib', exist_ok=True)
print('Setup complete.')

Setup complete.


## Step 1: Load graph dataset

In [2]:
with open(GRAPH_CACHE, 'rb') as f:
    graph_dataset = pickle.load(f)

# Drop any graphs with empty node feature matrices (defensive)
before = len(graph_dataset)
graph_dataset = [d for d in graph_dataset if d['data'].x.size(0) > 0]
n_empty = before - len(graph_dataset)

n_vuln = sum(1 for d in graph_dataset if d['label'] == 1)
print(f'Loaded {len(graph_dataset)} graphs  ({n_vuln} vulnerable)'
      + (f'  [{n_empty} empty graphs dropped]' if n_empty else ''))

/Users/xipingye/software-supply-chain-risk-detector/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 9668 graphs  (4823 vulnerable)


## Step 2: Extract raw root-node features

`build_pyg_data` guarantees the seed package is always placed at node index 0.
So `data.x[0]` gives its 27-dimensional raw feature vector directly, with no GNN involved.

In [3]:
features = []
labels   = []
pkg_ids  = []

for item in tqdm(graph_dataset, desc='Extracting root features'):
    features.append(item['data'].x[0].numpy())   # root node raw features (27-dim)
    labels.append(item['label'])
    pkg_ids.append(item['pkg_id'])

X_raw  = np.array(features) # (N, 27)
labels = np.array(labels)

print(f'Feature matrix : {X_raw.shape}')
print(f'Vulnerable     : {labels.sum()} / {len(labels)}')

Extracting root features: 100%|██████████| 9668/9668 [00:00<00:00, 468819.37it/s]

Feature matrix : (9668, 27)
Vulnerable     : 4823 / 9668


## Step 3: Train / test split

Uses the same seed and ratio as `train_gnn_model.ipynb` so the test sets are identical.

In [4]:
indices = list(range(len(graph_dataset)))
random.seed(42)
random.shuffle(indices)

split_idx    = int(len(indices) * TRAIN_RATIO)
test_indices = indices[split_idx:]

X_test      = X_raw[test_indices]
test_labels = labels[test_indices]
test_pkgids = [pkg_ids[i] for i in test_indices]

print(f'Test  : {len(X_test)}')
print(f'Vulnerable in test : {test_labels.sum()} / {len(test_labels)}')

Test  : 1934
Vulnerable in test : 989 / 1934


## Step 4: Standardise and K-Means — Elbow Method

In [5]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_test)

K_RANGE  = range(2, 21)
inertias = {}

for k in tqdm(K_RANGE, desc='Elbow sweep'):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias[k] = km.inertia_

elbow_df = pd.DataFrame({
    'K'      : list(inertias.keys()),
    'Inertia': list(inertias.values()),
})
elbow_df['Delta']    = elbow_df['Inertia'].diff().abs()
elbow_df['Delta2']   = elbow_df['Delta'].diff().abs()
elbow_df['Gain_pct'] = (elbow_df['Delta'] / elbow_df['Inertia'].shift(1) * 100).round(2)

print(elbow_df.to_string(index=False))

best_k = int(elbow_df.loc[elbow_df['Delta2'].idxmax(), 'K'])
print(f'\nAuto-detected elbow K = {best_k}')

Elbow sweep: 100%|██████████| 19/19 [00:00<00:00, 22.77it/s]

 K      Inertia       Delta      Delta2  Gain_pct
 2 45631.679688         NaN         NaN       NaN
 3 41072.773438 4558.906250         NaN      9.99
 4 38074.460938 2998.312500 1560.593750      7.30
 5 34986.726562 3087.734375   89.421875      8.11
 6 34414.828125  571.898438 2515.835938      1.63
 7 31180.109375 3234.718750 2662.820312      9.40
 8 29817.769531 1362.339844 1872.378906      4.37
 9 28018.724609 1799.044922  436.705078      6.03
10 26582.714844 1436.009766  363.035156      5.13
11 25214.351562 1368.363281   67.646484      5.15
12 23786.236328 1428.115234   59.751953      5.66
13 22990.261719  795.974609  632.140625      3.35
14 22102.488281  887.773438   91.798828      3.86
15 21088.890625 1013.597656  125.824219      4.59
16 19899.115234 1189.775391  176.177734      5.64
17 19983.093750   83.978516 1105.796875      0.42
18 18575.677734 1407.416016 1323.437500      7.04
19 17906.554688  669.123047  738.292969      3.60
20 17118.736328  787.818359  118.695312      4.40


In [6]:
# Override best_k here if you disagree with the auto-detection
# best_k = 10

final_km       = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = final_km.fit_predict(X_scaled)

results_df = pd.DataFrame({
    'pkg_id' : test_pkgids,
    'label'  : test_labels,
    'cluster': cluster_labels,
})

print(f'Trained KMeans with K={best_k}')
print(results_df.groupby('cluster')['label'].agg(['sum', 'count']).rename(
    columns={'sum': 'n_vulnerable', 'count': 'cluster_size'}
))

Trained KMeans with K=7
         n_vulnerable  cluster_size
cluster                            
0                  17           256
1                 274           375
2                 460           482
3                   3             5
4                 228           802
5                   7            13
6                   0             1


## Step 5: Evaluate

**Risk score** per cluster = `n_vulnerable / cluster_size`  
**Top-K clusters** = top 10% of clusters ranked by risk score

Metrics:
- **Recall** — fraction of all known-vulnerable packages that fall in the top-K clusters
- **Precision** — fraction of packages inside the top-K clusters that are actually vulnerable

In [7]:
cluster_stats = (
    results_df.groupby('cluster')['label']
    .agg(n_vulnerable='sum', cluster_size='count')
    .assign(risk_score=lambda df: df['n_vulnerable'] / df['cluster_size'])
    .sort_values('risk_score', ascending=False)
    .reset_index()
)

print('=== All clusters ranked by risk score ===')
print(cluster_stats.to_string(index=False))

n_top        = max(1, round(best_k * TOP_K_PCT))
top_clusters = set(cluster_stats.head(n_top)['cluster'])
print(f'\nTop-{n_top} cluster(s) (top {TOP_K_PCT*100:.0f}%): {top_clusters}')

total_vulnerable = (results_df['label'] == 1).sum()
in_top_mask      = results_df['cluster'].isin(top_clusters)
true_positives   = ((results_df['label'] == 1) & in_top_mask).sum()
top_cluster_size = in_top_mask.sum()

recall    = true_positives / total_vulnerable if total_vulnerable else 0.0
precision = true_positives / top_cluster_size if top_cluster_size  else 0.0
f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

print(f'\n--- Evaluation (top {TOP_K_PCT*100:.0f}% clusters as "risky") ---')
print(f'Total packages     : {len(results_df)}')
print(f'Total vulnerable   : {total_vulnerable}')
print(f'Top cluster size   : {top_cluster_size}')
print(f'True positives     : {true_positives}')
print(f'Recall             : {recall:.4f}  ({recall*100:.1f}%)')
print(f'Precision          : {precision:.4f}  ({precision*100:.1f}%)')
print(f'F1                 : {f1:.4f}')

=== All clusters ranked by risk score ===
 cluster  n_vulnerable  cluster_size  risk_score
       2           460           482    0.954357
       1           274           375    0.730667
       3             3             5    0.600000
       5             7            13    0.538462
       4           228           802    0.284289
       0            17           256    0.066406
       6             0             1    0.000000

Top-2 cluster(s) (top 33%): {1, 2}

--- Evaluation (top 33% clusters as "risky") ---
Total packages     : 1934
Total vulnerable   : 989
Top cluster size   : 857
True positives     : 734
Recall             : 0.7422  (74.2%)
Precision          : 0.8565  (85.6%)
F1                 : 0.7952


## Step 6: Save model

In [9]:
joblib.dump(final_km, MODEL_OUTPUT)
joblib.dump(scaler,   SCALER_OUTPUT)

cluster_stats.to_csv('../models/lib/raw_graph_cluster_risk_scores.csv', index=False)

print(f'KMeans saved      → {MODEL_OUTPUT}')
print(f'Scaler saved      → {SCALER_OUTPUT}')
print(f'Risk scores saved → ../models/lib/raw_graph_cluster_risk_scores.csv')

KMeans saved      → ../models/lib/raw_graph_kmeans.pkl
Scaler saved      → ../models/lib/raw_graph_scaler.pkl
Risk scores saved → ../models/lib/raw_graph_cluster_risk_scores.csv
